# Week 5 – Deep Learning Application in Data Science
## Handwritten Digit Classification using PyTorch

This notebook implements a complete deep-learning workflow on the publicly available **scikit-learn Digits dataset**. The task is ten-class classification of handwritten digits (0–9).


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
torch.manual_seed(42)
print("PyTorch version:", torch.__version__)


### 1. Load the public dataset

In [ ]:
digits = load_digits()
X = digits.data.astype(np.float32)
y = digits.target.astype(np.int64)

print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", np.unique(y))


In [ ]:
# Visualize a few original 8x8 digit images
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), digits.images[:10], digits.target[:10]):
    ax.imshow(image, cmap="gray")
    ax.set_title(f"Label: {label}")
    ax.axis("off")
plt.tight_layout()
plt.show()


### 2. Train/test split and preprocessing

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

print("Training samples:", X_train_s.shape[0])
print("Testing samples:", X_test_s.shape[0])


### 3. Network architecture

**64 inputs → 128 ReLU → Dropout(0.20) → 64 ReLU → Dropout(0.20) → 10 outputs**

The hidden layers learn non-linear representations. Dropout provides regularization, and the ten output logits correspond to the ten digit classes.


In [ ]:
class DigitsMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.network(x)

model = DigitsMLP()
print(model)


### 4. Training configuration

In [ ]:
train_ds = TensorDataset(torch.tensor(X_train_s), torch.tensor(y_train))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

print("Loss:", criterion)
print("Optimizer:", optimizer)


### 5. Train the neural network

In [ ]:
history = {"epoch": [], "train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}

for epoch in range(1, 31):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * len(yb)
        correct += (logits.argmax(1) == yb).sum().item()
        total += len(yb)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    with torch.no_grad():
        test_logits = model(torch.tensor(X_test_s))
        test_loss = criterion(test_logits, torch.tensor(y_test)).item()
        test_acc = (test_logits.argmax(1).numpy() == y_test).mean()

    history["epoch"].append(epoch)
    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    print(
        f"Epoch {epoch:02d}/30 | "
        f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | "
        f"Test loss: {test_loss:.4f} | Test acc: {test_acc:.4f}"
    )


### 6. Training curves

In [ ]:
hist = pd.DataFrame(history)

plt.figure(figsize=(8, 4))
plt.plot(hist["epoch"], hist["train_loss"], label="Train loss")
plt.plot(hist["epoch"], hist["test_loss"], label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Test Loss")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(hist["epoch"], hist["train_acc"], label="Train accuracy")
plt.plot(hist["epoch"], hist["test_acc"], label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training and Test Accuracy")
plt.legend()
plt.show()


### 7. Final evaluation

In [ ]:
model.eval()
with torch.no_grad():
    logits = model(torch.tensor(X_test_s))
    y_pred = logits.argmax(1).numpy()

accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, y_pred, average="weighted", zero_division=0
)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Weighted Precision", "Weighted Recall", "Weighted F1"],
    "Score": [accuracy, precision, recall, f1]
})
results


In [ ]:
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.arange(10))
disp.plot(values_format="d", cmap="Blues")
plt.title("Confusion Matrix – Test Set")
plt.show()


### 8. Critical analysis

The final metrics quantify overall classification quality, while the confusion matrix reveals class-specific mistakes. Dropout and weight decay were used to reduce overfitting risk. Standardization was fitted only on the training set to avoid leakage.

A key architectural limitation is that the 8×8 image is flattened into 64 values. A convolutional neural network would preserve spatial structure and is a natural next experiment.


### 9. Challenges and future improvements

- CPU-only execution: the compact dataset and MLP keep training practical.
- Overfitting: dropout, weight decay, and test monitoring were used.
- Representation: a CNN could learn spatial patterns more effectively.
- Tuning: future work can compare hidden-layer widths, learning rates, batch sizes, optimizers, and schedulers.
- Robustness: early stopping and repeated stratified evaluation could be added.


### 10. Conclusion

This notebook demonstrates the complete deep-learning workflow from public data acquisition and preprocessing to neural-network design, training, evaluation, and critical analysis.
